# Fase 2: DQN y PPO en Duckietown


En la Fase 1 resolvimos FrozenLake con Q-Learning tabular: 64 estados, una tabla que cabe entera en memoria. El problema de escalar es evidente: en Duckietown la observación es una imagen de cámara de 120×160×3. Eso equivale a un espacio de estados prácticamente infinito; no cabe ninguna tabla.

La solución es reemplazar la tabla por una **red neuronal** que aprenda a mapear imágenes a valores Q directamente. Eso es Deep RL, y es lo que hacemos en esta fase con dos algoritmos baseline:

| Algoritmo | Tipo | Acciones | Idea clave |
|-----------|------|----------|-----------|
| **DQN** | off-policy | discretas (5) | Replay buffer + red objetivo |
| **PPO** | on-policy | continuas | Objetivo recortado (clipped surrogate) |

Entrenamos sobre 5 mapas distintos para que el agente aprenda a generalizar en lugar de memorizar un circuito. El mapa de evaluación del profesor (`loop_obstacles`) no se usa en ningún momento.

> **Importante:** ejecutar en **Google Colab con GPU activada** (`Entorno de ejecución → T4 GPU`). Duckietown usa OpenGL para renderizar, que no funciona en Windows sin display virtual.


---

### Setup (Python 3.11 + dependencias)

`gym-duckietown` es nativo de **Python 3.11**, pero Colab trae 3.12. Por eso instalamos Python 3.11 del sistema (vía *deadsnakes*) y ejecutamos el entrenamiento como un **script** con ese intérprete, lanzado con `xvfb-run`. Así el render OpenGL corre en un proceso aparte con su propio contexto y **no se cae el kernel** (en Colab el síntoma del fallo era `restarting kernel`). En local estas celdas solo informan; Duckietown necesita Linux/WSL2.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = "Aprendizaje-por-Refuerzo-y-Conducci-n-Aut-noma"
    # situarse en la raiz del repo (clonar si hace falta) para tener requirements.txt y src/
    if os.path.basename(os.getcwd()) != REPO:
        if not os.path.exists(REPO):
            os.system(f"git clone https://github.com/JaviCeronn/{REPO}.git")
        os.chdir(REPO)
    os.system("git pull origin main")

    # Python 3.11 del SISTEMA (deadsnakes). gym-duckietown es nativo de 3.11; en el 3.12 de
    # Colab no se puede renderizar Duckietown dentro del kernel sin segfaultearlo. NO usar
    # conda: sobreescribe libEGL de NVIDIA y rompe el render en GPU. El entrenamiento se
    # ejecuta como SCRIPT con este 3.11.
    os.system("sudo apt-get install -y -qq software-properties-common > /dev/null")
    os.system("sudo add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1")
    os.system("sudo apt-get update -qq")
    os.system("sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev python3.11-distutils > /dev/null")
    os.system("wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py")
    os.system("python3.11 /tmp/get-pip.py -q")
    PY = "/usr/bin/python3.11"
    os.system(f"{PY} --version")
    print(">>> Python 3.11 listo <<<")
else:
    PY = sys.executable
    print("Local: las Fases 2-3 (Duckietown) requieren Linux/WSL2 + Python 3.11.")
    print("       En Windows solo corre la Fase 1. Usa Google Colab con GPU.")

REPO_ROOT = os.getcwd()
print("Interprete RL :", PY)
print("Raiz repo     :", REPO_ROOT)

In [ ]:
# Dependencias del SCRIPT de entrenamiento (Python 3.11), NO del kernel del notebook.
# requirements.txt fija el stack para 3.11 (numpy 1.23.5, gym 0.25.2, SB3 2.2.1, torch 2.12
# y el ecosistema daffy/zuper). gym-duckietown se instala aparte con --no-deps (su setup.py
# fija numpy<=1.20, sin wheels para 3.11). Tarda ~5-10 min la primera vez.
if IN_COLAB:
    os.system("sudo apt-get install -y -qq xvfb freeglut3-dev libosmesa6-dev "
              "libgl1-mesa-dri libgl1-mesa-glx libglu1-mesa libturbojpeg > /dev/null")
    os.system(f"{PY} -m pip install -q -r requirements.txt")
    os.system(f"{PY} -m pip install -q --no-deps git+https://github.com/duckietown/gym-duckietown.git@daffy")
    os.system(f'{PY} -c "import gym_duckietown, numpy; print(\'gym_duckietown OK, numpy\', numpy.__version__)"')
    print(">>> Dependencias (Python 3.11) listas <<<")
else:
    print("Instala una vez en tu terminal Linux/WSL con Python 3.11:")
    print("  python3.11 -m pip install -r requirements.txt")
    print("  python3.11 -m pip install --no-deps git+https://github.com/duckietown/gym-duckietown.git@daffy")

---

### Preprocesado y arquitectura

Antes de entrenar resolvemos dos problemas: el entorno usa la API antigua de `gym` y produce imágenes de 120×160×3; SB3 espera la API `gymnasium` y tensores compactos.

**`DuckieWrapper`** — adapta la API y preprocesa la observación:
1. Recortar el 50 % superior (el cielo no aporta información para seguir el carril)
2. Convertir a escala de grises (la geometría de la carretera no necesita color)
3. Redimensionar a 64×64 (suficiente detalle, mucho más rápido de procesar)

**Frame stacking (4 frames):** un frame estático no contiene información de movimiento. Al apilar 4 frames consecutivos la observación final es `(4, 64, 64)` y el agente puede inferir velocidad y dirección del giro.

**`DiscreteWrapper`** — DQN exige acciones discretas. Mapeamos 5 combinaciones de velocidades de rueda al espacio `{0..4}`. Es una simplificación: el agente no puede ejecutar giros de precisión arbitraria.

**`CustomCNN`** (Nature CNN, Mnih 2015): tres capas convolucionales que extraen la geometría de la carretera, seguidas de una capa lineal que produce un vector de 256 dimensiones. Ese vector alimenta las cabezas de política/valor de DQN y PPO.


---

### ¿Qué ve la cámara?

Visualizamos el pipeline de visión paso a paso con una imagen sintética de carretera. Funciona en local sin necesitar el simulador — en Colab se puede sustituir por un frame real de Duckietown.


In [ ]:
import os, pathlib
from IPython.display import Image, display

RESULTS = pathlib.Path("results") / "Fase_2"
RESULTS.mkdir(parents=True, exist_ok=True)
vis = RESULTS / "vision_pipeline.png"

try:
    IN_COLAB
except NameError:
    IN_COLAB = "google.colab" in __import__("sys").modules

if IN_COLAB:
    # Frame REAL de Duckietown: lo captura el script en un subproceso (py3.11 + xvfb),
    # porque crear el simulador dentro del kernel lo segfaultea.
    os.system(f'xvfb-run -a -s "-screen 0 1024x768x24" {PY} src/train_fase2.py --algo frame')
    if vis.exists():
        display(Image(str(vis)))
    else:
        print("No se pudo capturar el frame real; revisa el Paso de instalacion.")
else:
    # Local (sin simulador): imagen sintetica que imita un frame de Duckietown.
    import numpy as np
    import cv2
    import matplotlib.pyplot as plt

    # imagen sintetica que imita un frame tipico de Duckietown (120x160x3 RGB)
    H, W = 120, 160
    frame = np.zeros((H, W, 3), dtype=np.uint8)

    # cielo (azul claro) en la mitad superior
    frame[:H//2, :] = [135, 185, 210]
    # cesped a los lados de la carretera
    frame[H//2:, :30]  = [60, 120, 50]
    frame[H//2:, 130:] = [60, 120, 50]
    # asfalto (gris oscuro)
    frame[H//2:, 30:130] = [70, 70, 70]
    # linea central amarilla discontinua
    for y in range(H//2, H, 16):
        frame[y:y+8, 76:84] = [230, 200, 30]
    # bordillos blancos
    frame[H//2:, 30:34]   = [220, 220, 220]
    frame[H//2:, 126:130] = [220, 220, 220]

    # --- pipeline paso a paso ---
    cropped = frame[H//2:, :, :]                                           # paso 1: recortar cielo
    gray    = cv2.cvtColor(cropped, cv2.COLOR_RGB2GRAY)                    # paso 2: grises
    resized = cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA)     # paso 3: 64x64

    # paso 4: simular 4 frames consecutivos (el coche avanza ligeramente entre ellos)
    frames_stack = []
    for shift in [6, 4, 2, 0]:
        f = np.roll(resized, -shift, axis=0).copy()
        if shift > 0:
            f[-shift:, :] = resized[-1, :]
        frames_stack.append(f)

    # --- figura ---
    fig = plt.figure(figsize=(15, 5))
    fig.suptitle("Pipeline de vision — lo que ve la camara de Duckietown", fontweight="bold", fontsize=12)

    steps   = ["1. Frame original\n120x160  RGB", "2. Recorte 50% superior\n60x160  RGB",
               "3. Escala de grises\n60x160  1 canal", "4. Resize 64x64\n64x64  1 canal"]
    imgs    = [frame, cropped, gray, resized]
    cmaps   = [None, None, "gray", "gray"]

    for i, (img, title, cmap) in enumerate(zip(imgs, steps, cmaps), 1):
        ax = fig.add_subplot(2, 5, i)
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=8)
        ax.axis("off")

    for j, f in enumerate(frames_stack):
        ax = fig.add_subplot(2, 5, 6 + j)
        ax.imshow(f, cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"5. Frame t-{3-j}  (stack {j+1}/4)\n64x64  1 canal", fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    p = RESULTS / "vision_pipeline.png"
    plt.savefig(p, dpi=130, bbox_inches="tight"); plt.close()
    print(f"Guardado: {p}"); display(Image(str(p)))
    print(f"\nForma final del tensor: {len(frames_stack)} x 64 x 64  ->  (4, 64, 64)")
    print("CustomCNN recibe un batch de forma (N, 4, 64, 64)")

---

### Pipeline de entrenamiento

La siguiente celda **escribe** `src/train_fase2.py`: contiene las clases que acabamos de describir (`DuckieWrapper`, `DiscreteWrapper`, `LaneFollowingReward`, `CustomCNN`) y los entrenamientos de **DQN** y **PPO**, además de la evaluación comparativa. Cada bloque se invoca por separado con `--algo {frame,dqn,ppo,eval}`.

Va en un script (y no en celdas) por un motivo técnico de `gym-duckietown`: mantiene estado OpenGL global por proceso, así que renderizar dentro del kernel del notebook lo segfaultea. Como script lanzado con `xvfb-run`, cada simulador vive en su propio proceso (`SubprocVecEnv`, que además puede reimportar el módulo) y todo es estable.

In [ ]:
%%writefile src/train_fase2.py
"""
FASE 2 - Baselines DQN y PPO en Duckietown.

Se ejecuta como SCRIPT con Python 3.11 + xvfb-run, NO dentro del kernel de Jupyter:
gym-duckietown mantiene estado OpenGL GLOBAL por proceso y SEGFAULTEA el proceso si se
renderiza dentro del kernel del notebook (en Colab se ve como "restarting kernel"). Al
lanzarlo como proceso independiente con xvfb-run, cada simulador tiene su contexto GL y el
entrenamiento es estable; ademas SubprocVecEnv (spawn) puede reimportar este modulo .py.

Subcomandos (--algo):
    frame  captura un frame REAL de Duckietown y guarda el pipeline de vision
    dqn    entrena DQN (baseline discreto), guarda modelo + curva
    ppo    entrena PPO (baseline continuo), guarda modelo + curva
    eval   carga DQN y PPO ya entrenados, evalua, guarda grafica comparativa y best_agent.zip
    all    dqn + ppo + eval

Uso:  python src/train_fase2.py --algo dqn --timesteps 30000
"""
from __future__ import annotations
import argparse
import logging
import os
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"     # lo heredan los subprocesos 'spawn' de SubprocVecEnv
warnings.filterwarnings("ignore")
logging.disable(logging.INFO)               # corte GLOBAL: el stack daffy no lo revierte

import numpy as np
import gym as old_gym                        # API antigua (gym-duckietown)
import gymnasium as gym                      # API moderna (Stable-Baselines3)
from gymnasium import spaces
import cv2
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecFrameStack, VecMonitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import BaseCallback

IMG_SIZE = 64
N_STACK = 4
OBS_SHAPE = (1, IMG_SIZE, IMG_SIZE)
SEED = 42
RESULTS = os.path.join("results", "Fase_2")
os.makedirs(RESULTS, exist_ok=True)

DQN_PATH = os.path.join(RESULTS, "dqn_duckie")
PPO_PATH = os.path.join(RESULTS, "ppo_duckie")

TRAIN_MAPS = [
    "Duckietown-loop_empty-v0",
    "Duckietown-udem1-v0",
    "Duckietown-zigzag_dists-v0",
    "Duckietown-small_loop-v0",
    "Duckietown-straight_road-v0",
]

DISCRETE_ACTIONS = np.array([
    [0.6, 0.6],    # 0 recto
    [0.35, 0.6],   # 1 giro suave izquierda
    [0.6, 0.35],   # 2 giro suave derecha
    [0.2, 0.6],    # 3 giro fuerte izquierda
    [0.6, 0.2],    # 4 giro fuerte derecha
], dtype=np.float32)


class DuckieWrapper(gym.Env):
    """Adapta gym-duckietown a Gymnasium: recorta cielo, gris, 64x64 -> (1,64,64)."""
    metadata = {"render_modes": ["rgb_array"]}

    def __init__(self, env_name="Duckietown-loop_empty-v0", seed=None):
        super().__init__()
        import gym_duckietown                 # registra los entornos al importar
        logging.disable(logging.INFO)         # el stack daffy re-activa sus loggers al importar
        self.env_name = env_name
        self.env = old_gym.make(env_name)
        if seed is not None:
            try:
                self.env.seed(seed)
            except Exception:
                pass
        self.action_space = spaces.Box(
            low=np.array([-1.0, -1.0], dtype=np.float32),
            high=np.array([1.0, 1.0], dtype=np.float32), dtype=np.float32)
        self.observation_space = spaces.Box(low=0, high=255, shape=OBS_SHAPE, dtype=np.uint8)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs = self.env.reset()
        if isinstance(obs, tuple):
            obs = obs[0]
        return self._process_obs(obs), {}

    def step(self, action):
        action = np.asarray(action, dtype=np.float32).reshape(-1)
        obs, reward, done, info = self.env.step(action)
        return self._process_obs(obs), float(reward), bool(done), False, info

    def _process_obs(self, obs):
        obs = obs[obs.shape[0] // 2:, :, :]                                   # recortar cielo
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)                          # escala de grises
        resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        return np.expand_dims(resized, axis=0).astype(np.uint8)              # (1,64,64)

    def render(self):
        return self.env.render(mode="rgb_array")

    def close(self):
        self.env.close()


class DiscreteWrapper(gym.ActionWrapper):
    """Discretiza el espacio de accion continuo para DQN (5 acciones)."""
    def __init__(self, env):
        super().__init__(env)
        self.action_space = spaces.Discrete(len(DISCRETE_ACTIONS))

    def action(self, action):
        return DISCRETE_ACTIONS[int(action)]


class LaneFollowingReward(gym.Wrapper):
    """Reward shaping (solo entrenamiento): penaliza salirse (-10), premia avanzar (+0.1)."""
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        shaped = reward
        if terminated and reward < 0:
            shaped = -10.0
        shaped += 0.1
        return obs, shaped, terminated, truncated, info


class CustomCNN(BaseFeaturesExtractor):
    """CNN tipo Nature (Mnih 2015) adaptada a (4,64,64) con normalizacion de pixeles."""
    def __init__(self, observation_space: spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            sample = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        return self.linear(self.cnn(observations.float() / 255.0))


def make_env(map_name, discrete=False, shaping=True, seed=SEED):
    def _init():
        env = DuckieWrapper(map_name, seed=seed)
        if shaping:
            env = LaneFollowingReward(env)
        if discrete:
            env = DiscreteWrapper(env)
        return env
    return _init


def make_vec_env(maps, discrete=False, shaping=True, n_stack=N_STACK):
    # gym-duckietown guarda estado OpenGL GLOBAL por proceso: tener mas de un simulador en el
    # MISMO proceso (DummyVecEnv) provoca un Segmentation fault al renderizar el segundo
    # contexto. SubprocVecEnv aisla cada entorno en su propio proceso (un contexto GL
    # independiente); 'spawn' evita conflictos con el contexto CUDA del proceso padre.
    # Con un solo mapa basta DummyVecEnv.
    env_fns = [make_env(m, discrete=discrete, shaping=shaping, seed=SEED + i)
               for i, m in enumerate(maps)]
    if len(env_fns) == 1:
        vec = DummyVecEnv(env_fns)
    else:
        vec = SubprocVecEnv(env_fns, start_method="spawn")
    vec = VecFrameStack(vec, n_stack=n_stack)
    vec = VecMonitor(vec)
    return vec


def policy_kwargs(features_dim=256):
    return dict(features_extractor_class=CustomCNN,
                features_extractor_kwargs=dict(features_dim=features_dim))


class RewardLogger(BaseCallback):
    """Recoge la recompensa de cada episodio (VecMonitor la pone en info['episode']['r'])."""
    def __init__(self):
        super().__init__(verbose=0)
        self.ep_rewards = []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.ep_rewards.append(info["episode"]["r"])
        return True


def plot_curve(rewards, color, title, filename):
    if not rewards:
        print(f"[aviso] sin episodios completos para {filename}")
        return
    w = min(20, len(rewards))
    ma = np.convolve(rewards, np.ones(w) / w, mode="valid")
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(ma, color=color)
    ax.axhline(ma[-1], color="tomato", linestyle="--", label=f"Final: {ma[-1]:.1f}")
    ax.set(xlabel="Episodio", ylabel=f"Recompensa (media movil {w})", title=title)
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
    path = os.path.join(RESULTS, filename)
    plt.savefig(path, dpi=120, bbox_inches="tight"); plt.close()
    print(f"Guardado: {path}")


# ------------------------------------------------------------------ FRAME REAL
def capture_frame():
    """Captura 4 frames REALES de Duckietown y guarda el pipeline de vision."""
    import gym_duckietown  # noqa: F401
    env = old_gym.make("Duckietown-loop_empty-v0")
    obs = env.reset()
    if isinstance(obs, tuple):
        obs = obs[0]
    raw = [np.asarray(obs)]
    for _ in range(3):                                   # avanzar recto para 4 frames distintos
        out = env.step(np.array([0.5, 0.5], dtype=np.float32))
        o = out[0]
        raw.append(np.asarray(o))
    env.close()

    frame = raw[0]
    H = frame.shape[0]
    cropped = frame[H // 2:, :, :]
    gray = cv2.cvtColor(cropped, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    stack = []
    for fr in raw:
        c = fr[fr.shape[0] // 2:, :, :]
        g = cv2.cvtColor(c, cv2.COLOR_RGB2GRAY)
        stack.append(cv2.resize(g, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA))

    fig = plt.figure(figsize=(15, 5))
    fig.suptitle("Pipeline de vision - frame REAL de Duckietown", fontweight="bold", fontsize=12)
    h0, w0 = frame.shape[0], frame.shape[1]
    steps = [f"1. Frame original\n{h0}x{w0}  RGB", f"2. Recorte 50% superior\n{h0//2}x{w0}  RGB",
             f"3. Escala de grises\n{h0//2}x{w0}  1 canal", "4. Resize 64x64\n64x64  1 canal"]
    imgs = [frame, cropped, gray, resized]
    cmaps = [None, None, "gray", "gray"]
    for i, (img, title, cmap) in enumerate(zip(imgs, steps, cmaps), 1):
        ax = fig.add_subplot(2, 5, i)
        ax.imshow(img, cmap=cmap); ax.set_title(title, fontsize=8); ax.axis("off")
    for j, f in enumerate(stack):
        ax = fig.add_subplot(2, 5, 6 + j)
        ax.imshow(f, cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"5. Frame t-{3-j}  (stack {j+1}/4)\n64x64  1 canal", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    path = os.path.join(RESULTS, "vision_pipeline.png")
    plt.savefig(path, dpi=130, bbox_inches="tight"); plt.close()
    print(f"Guardado: {path}")


# ------------------------------------------------------------------ ENTRENAMIENTO
def train_dqn(timesteps, device):
    print("\n=== Entrenando DQN (baseline discreto) ===")
    env = make_vec_env(TRAIN_MAPS, discrete=True, shaping=True)
    model = DQN("CnnPolicy", env, policy_kwargs=policy_kwargs(),
                learning_rate=1e-4, buffer_size=50_000, learning_starts=1_000,
                batch_size=64, gamma=0.99, train_freq=4, target_update_interval=1_000,
                exploration_fraction=0.2, exploration_final_eps=0.05,
                verbose=1, seed=SEED, device=device)
    cb = RewardLogger()
    model.learn(total_timesteps=timesteps, callback=cb)
    model.save(DQN_PATH); env.close()
    plot_curve(cb.ep_rewards, "steelblue", "DQN - curva de entrenamiento en Duckietown",
               "dqn_training.png")
    print(f"Modelo guardado: {DQN_PATH}.zip")


def train_ppo(timesteps, device):
    print("\n=== Entrenando PPO (baseline continuo) ===")
    env = make_vec_env(TRAIN_MAPS, discrete=False, shaping=True)
    model = PPO("CnnPolicy", env, policy_kwargs=policy_kwargs(),
                learning_rate=3e-4, n_steps=2_048, batch_size=256, n_epochs=10,
                gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
                verbose=1, seed=SEED, device=device)
    cb = RewardLogger()
    model.learn(total_timesteps=timesteps, callback=cb)
    model.save(PPO_PATH); env.close()
    plot_curve(cb.ep_rewards, "seagreen", "PPO - curva de entrenamiento en Duckietown",
               "ppo_training.png")
    print(f"Modelo guardado: {PPO_PATH}.zip")


def evaluate_agent(model, eval_maps, n_episodes=5, discrete=False):
    env = make_vec_env(eval_maps, discrete=discrete, shaping=False)
    returns = []
    for _ in range(n_episodes):
        obs = env.reset()
        done = np.array([False])
        ep_ret, steps = 0.0, 0
        while not done[0] and steps < 1000:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _ = env.step(action)
            ep_ret += float(reward[0]); steps += 1
        returns.append(ep_ret)
    env.close()
    return float(np.mean(returns)), float(np.std(returns)), returns


def plot_comparison(dqn_stats, ppo_stats):
    dqn_mean, dqn_std, _ = dqn_stats
    ppo_mean, ppo_std, _ = ppo_stats
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(["DQN (discreto)", "PPO (continuo)"], [dqn_mean, ppo_mean],
                  yerr=[dqn_std, ppo_std], capsize=8,
                  color=["steelblue", "seagreen"], alpha=0.85)
    for bar, m in zip(bars, [dqn_mean, ppo_mean]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(dqn_std, ppo_std) * 0.15 + 0.5,
                f"{m:.1f}", ha="center", va="bottom", fontweight="bold")
    ax.set(ylabel="Recompensa media (5 episodios)",
           title="DQN vs PPO - evaluacion en los 5 mapas de entrenamiento")
    ax.grid(axis="y", alpha=0.3); plt.tight_layout()
    path = os.path.join(RESULTS, "dqn_vs_ppo.png")
    plt.savefig(path, dpi=120, bbox_inches="tight"); plt.close()
    print(f"Guardado: {path}")


def run_eval(device, n_episodes):
    print("\n=== Evaluacion comparativa (modo determinista) ===")
    dqn_model = DQN.load(DQN_PATH, device=device)
    ppo_model = PPO.load(PPO_PATH, device=device)
    dqn_stats = evaluate_agent(dqn_model, TRAIN_MAPS, n_episodes, discrete=True)
    ppo_stats = evaluate_agent(ppo_model, TRAIN_MAPS, n_episodes, discrete=False)
    plot_comparison(dqn_stats, ppo_stats)

    # mejor agente continuo -> entregable. Guardamos los dos nombres en uso:
    #   best_agent.zip        (requisitos de entrega / diapositiva 9)
    #   best_duckie_agent.zip (el que carga notebooks/eval.ipynb del profesor)
    ppo_model.save("best_agent")
    ppo_model.save("best_duckie_agent")
    print("\nbest_agent.zip y best_duckie_agent.zip guardados.")
    print(f"DQN: {dqn_stats[0]:.2f} +/- {dqn_stats[1]:.2f} | retornos: {[round(x,1) for x in dqn_stats[2]]}")
    print(f"PPO: {ppo_stats[0]:.2f} +/- {ppo_stats[1]:.2f} | retornos: {[round(x,1) for x in ppo_stats[2]]}")


def main():
    parser = argparse.ArgumentParser(description="Fase 2 - DQN y PPO en Duckietown.")
    parser.add_argument("--algo", choices=["frame", "dqn", "ppo", "eval", "all"], default="all")
    parser.add_argument("--timesteps", type=int, default=30_000)
    parser.add_argument("--eval-episodes", type=int, default=5)
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Dispositivo:", device)

    if args.algo == "frame":
        capture_frame()
        return
    if args.algo in ("dqn", "all"):
        train_dqn(args.timesteps, device)
    if args.algo in ("ppo", "all"):
        train_ppo(args.timesteps, device)
    if args.algo in ("eval", "all"):
        run_eval(device, args.eval_episodes)


if __name__ == "__main__":
    main()

---

### DQN (baseline discreto)

**Deep Q-Network** es la traducción directa del Q-Learning tabular de la Fase 1 al mundo de los píxeles. Allí guardábamos un número por cada par `(estado, acción)` en una tabla de 64×4; aquí eso es imposible ( hay infinitas imágenes de cámara posibles ), así que sustituimos la tabla por una **red neuronal** que aprende la función de valor `Q(s, ·)`: recibe el stack de 4 frames y devuelve de una sola pasada los 5 valores Q, uno por cada acción discreta.

**Cómo elige la acción:** exactamente igual que en la Fase 1, con una política **ε-greedy**. Con probabilidad ε prueba una acción al azar; el resto del tiempo explota lo aprendido eligiendo `argmax_a Q(s,a)`, la acción que la red estima más valiosa. ε arranca alto y decae (lo controla `exploration_fraction`): el coche explora mucho al principio y, cuando ya sabe conducir, se ciñe a lo que funciona. El descuento `γ=0.99` hace que valore casi tanto las recompensas futuras como las inmediatas, lo que lo empuja a mantenerse en el carril a largo plazo en vez de buscar solo el premio del paso siguiente.

**Por qué hacen falta dos trucos:** entrenar una red contra el objetivo `r + γ·max Q(s',·)` es inestable, porque cada actualización mueve a la vez la predicción y el propio objetivo que persigue. DQN lo estabiliza con dos mecanismos:

- **Replay buffer:** almacenamos las transiciones vividas y entrenamos muestreando mini-batches aleatorios del pasado, en lugar de aprender de cada transición en el momento. Esto rompe la **correlación temporal** entre frames consecutivos (que son casi idénticos y sesgarían el gradiente) y, al reutilizar cada experiencia muchas veces, hace a DQN **eficiente en muestras**. Solo es posible porque DQN es **off-policy**: puede aprender de datos generados por una política anterior.
- **Red objetivo** (target network): mantenemos una copia congelada de la red Q para calcular el target y la sincronizamos solo cada `target_update_interval` pasos. Sin esto, el objetivo cambiaría en cada actualización (la red "perseguiría su propia cola" )y el aprendizaje diverge.

La limitación de fondo es que DQN exige acciones **discretas**. Con 5 combinaciones fijas de velocidades de rueda el agente nunca puede hacer un giro de precisión arbitraria: tiene que aproximarlo encadenando giros suaves y fuertes, y eso se nota sobre todo en curvas cerradas.

Hiperparámetros: `lr=1e-4`, `buffer=50k`, `batch=64`, `γ=0.99`, `exploration_fraction=0.2`.


In [ ]:
# Entrena DQN como script con Python 3.11 bajo xvfb-run (proceso aparte -> no crashea el
# kernel). Empieza con pocos timesteps para validar; sube a 200000+ para resultados reales.
TIMESTEPS = 30000

if IN_COLAB:
    os.system(f'xvfb-run -a -s "-screen 0 1024x768x24" {PY} src/train_fase2.py --algo dqn --timesteps {TIMESTEPS}')
    from IPython.display import Image, display
    p = os.path.join("results", "Fase_2", "dqn_training.png")
    display(Image(p)) if os.path.exists(p) else print("No se genero la curva DQN.")
else:
    print("Ejecuta en Colab. En Windows no hay EGL/OpenGL para Duckietown.")

**¿Qué vemos?**

Al principio la recompensa es muy negativa: el agente sale de la carretera constantemente y acumula penalizaciones de -10. A medida que el buffer se llena y ε decrece, el agente aprende a mantenerse en el carril y la recompensa sube. Con 30k pasos vemos el inicio del aprendizaje para convergencia real hacen falta 200k+ en GPU.

La limitación de las 5 acciones discretas se nota en curvas cerradas: el agente tiene que encadenar varios giros fuertes donde un control continuo haría un solo movimiento progresivo.


---

### PPO (baseline continuo)

Mientras DQN aprende *valores* y deriva la política de ellos (el `argmax`), **Proximal Policy Optimization** aprende la **política directamente**: la red produce una distribución sobre acciones continuas —una gaussiana sobre las dos velocidades de rueda, con su media y su desviación— y se entrena por **ascenso de gradiente** para subir la probabilidad de las acciones que salieron bien. Por eso opera de forma nativa en el espacio continuo y no necesita el `DiscreteWrapper`.

**Cómo aprende, paso a paso:** (1) recoge `n_steps=2048` transiciones con la política actual; (2) estima cuánto mejor o peor fue cada acción respecto a lo esperado mediante la **ventaja** `A_t` (con GAE, `gae_λ=0.95`, que equilibra sesgo y varianza); (3) hace `n_epochs=10` pasadas de gradiente sobre esos datos. Su innovación es el **objetivo recortado** (*clipped surrogate*):

```
L_CLIP = E[ min( r_t · A_t ,  clip(r_t, 1−ε, 1+ε) · A_t ) ]
```

El ratio `r_t = π(a|s) / π_old(a|s)` mide cuánto ha cambiado la política desde la última actualización. **Por qué el clip:** al recortar `r_t` a la banda `[1−ε, 1+ε]` se elimina el incentivo a moverse demasiado lejos en un solo paso; equivale a una región de confianza barata que evita las actualizaciones destructivas típicas del gradiente de política y hace a PPO muy estable en la práctica. El término `ent_coef=0.01` premia mantener algo de aleatoriedad en la gaussiana, de modo que el agente siga explorando y no colapse pronto en una conducción rígida.

Con acciones continuas el agente ejecuta exactamente el giro que necesita, produciendo una conducción más suave que DQN. El precio es que PPO es **on-policy**: tras cada actualización descarta toda la experiencia y vuelve a recolectar con la política nueva, lo que lo hace menos eficiente en muestras que DQN.

Hiperparámetros: `lr=3e-4`, `n_steps=2048`, `batch=256`, `n_epochs=10`, `γ=0.99`, `gae_λ=0.95`, `clip=0.2`, `ent_coef=0.01`.


In [ ]:
# Entrena PPO como script con Python 3.11 bajo xvfb-run (mismo patron que DQN).
TIMESTEPS = 30000

if IN_COLAB:
    os.system(f'xvfb-run -a -s "-screen 0 1024x768x24" {PY} src/train_fase2.py --algo ppo --timesteps {TIMESTEPS}')
    from IPython.display import Image, display
    p = os.path.join("results", "Fase_2", "ppo_training.png")
    display(Image(p)) if os.path.exists(p) else print("No se genero la curva PPO.")
else:
    print("Ejecuta en Colab. En Windows no hay EGL/OpenGL para Duckietown.")

**¿Qué vemos?**

PPO suele mostrar una curva más suave que DQN porque el gradiente de política es más estable. Al tener acciones continuas el agente puede ajustar gradualmente sus giros en lugar de saltar entre categorías discretas. La contrapartida es que necesita acumular `n_steps=2048` experiencias antes de cada actualización, lo que puede hacer que arranque más lento con el mismo presupuesto de pasos.


---

### Evaluación comparativa

Evaluamos ambos modelos en modo determinista (sin exploración) sobre los 5 mapas de entrenamiento. El mejor agente continuo se guarda como `best_agent.zip` (requisito de entrega) y también como `best_duckie_agent.zip` (el nombre que carga `eval.ipynb` del profesor).


In [ ]:
# Carga los dos modelos ya entrenados, los evalua en modo determinista sobre los 5 mapas y
# genera la grafica comparativa DQN vs PPO. Guarda el mejor agente continuo como best_agent.zip.
if IN_COLAB:
    os.system(f'xvfb-run -a -s "-screen 0 1024x768x24" {PY} src/train_fase2.py --algo eval')
    from IPython.display import Image, display
    p = os.path.join("results", "Fase_2", "dqn_vs_ppo.png")
    display(Image(p)) if os.path.exists(p) else print("No se genero la grafica comparativa.")
else:
    print("Ejecuta en Colab. En Windows no hay EGL/OpenGL para Duckietown.")

**Conclusiones de la Fase 2**

Esta fase demuestra que es posible aprender a conducir desde píxeles: algo impensable con la Q-Table de la Fase 1. Con 30k pasos vemos los primeros pasos del aprendizaje, no el rendimiento final.

Las diferencias estructurales entre los dos algoritmos ya se aprecian aunque no haya convergencia:

- **DQN** arranca más rápido gracias al replay buffer, pero su techo está limitado por la discretización. En curvas cerradas tiene que encadenar varias acciones discretas donde un control continuo haría un solo movimiento progresivo.
- **PPO** produce conducción más suave al operar en el espacio continuo, pero al ser on-policy necesita más pasos para aprovechar cada experiencia.

Estos dos resultados son la **referencia** para la Fase 3. La hipótesis es que **SAC** superará a PPO al ser off-policy (reutiliza experiencias como DQN) y a DQN al usar control continuo (como PPO), añadiendo exploración por máxima entropía que genera políticas más robustas en el mapa oculto.
